# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    plot_mwave_stabilization_groups, compute_mwave_size,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    compute_peri_stim_bg, PERI_STIM_BG_START_MS, PERI_STIM_BG_END_MS,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials, make_failed_trials_viewer,
    exclude_failed_trials, filter_recordings_by_metric,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups, session_trial_numbers, build_trial_number_groups,
    make_viewer, make_ft_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    load_all_recordings, FT_SNAP_HZ, build_settings_dataframe,
    # Save / load an analysis snapshot
    build_trial_dataframe, collect_notebook_config, diff_analysis_config,
    save_analysis_pickle, load_analysis_pickle, merge_recording_sets,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [2]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    
    

    # HRPilot-27 Recordings

    #("-5",  "Control_Mode/HRPilot-27/NEW_SET-UP_TEST_HRPILOT-27_BOOTH4_100US_10KHZ_10-16-26",        10000.0),
    #("-4",  "Control_Mode/HRPilot-27/CMN2_HRPILOT-27_BOOTH4_100US_10KHZ_10-20-26",        10000.0),
    #("-3",  "Control_Mode/HRPilot-27/CMN3_HRPILOT-27_BOOTH4_100US_10KHZ_10-21-26",        10000.0),
    #("-2",  "Control_Mode/HRPilot-27/CMN4_HRPILOT-27_BOOTH4_100US_10KHZ_10-21-26",        10000.0),
    #("-1",  "Control_Mode/HRPilot-27/CMN5_HRPILOT-27_BOOTH4_100US_10KHZ_10-22-26",        10000.0),
    
    #("1",  "Up-Conditioning_VNS/HRPilot-27/UPVNS1_HRPILOT-27_BOOTH4_100US_10KHZ_10-22-26",        10000.0),
    #("2",  "Up-Conditioning_VNS/HRPilot-27/UPVNS2.5_BOTH_HRPILOT-27_BOOTH4_100US_10KHZ_10-23-26",        10000.0),
    #("3",  "Up-Conditioning_VNS/HRPilot-27/UPVNS4_HRPILOT-27_BOOTH4_100US_10KHZ_10-26-26",        10000.0),
    #("4",  "Up-Conditioning_VNS/HRPilot-27/UPVNS5_HRPILOT-27_BOOTH4_100US_10KHZ_10-27-26",        10000.0),
    #("5",  "Up-Conditioning_VNS/HRPilot-27/UPVNS6_HRPILOT-27_BOOTH4_100US_10KHZ_10-28-26",        10000.0),
    #("6",  "Up-Conditioning_VNS/HRPilot-27/UPVNS7_HRPILOT-27_BOOTH4_100US_10KHZ_10-29-26",        10000.0),
    
    ("7",  "Up-Conditioning_VNS/HRPilot-27/UPVNS8_HRPILOT-27_BOOTH4_100US_10KHZ_10-30-26",        10000.0),
    ("8",  "Up-Conditioning_VNS/HRPilot-27/UPVNS9_HRPILOT-27_BOOTH4_100US_10KHZ_10-31-26",        10000.0),
    ("9",  "Up-Conditioning_VNS/HRPilot-27/UPVNS10_HRPILOT-27_BOOTH4_100US_10KHZ_11-2-26",        10000.0),
    ("10",  "Up-Conditioning_VNS/HRPilot-27/UPVNS11_HRPILOT-27_BOOTH4_100US_10KHZ_11-3-26",        10000.0),
    ("11",  "Up-Conditioning_VNS/HRPilot-27/UPVNS12_HRPILOT-27_BOOTH4_100US_10KHZ_11-4-26",        10000.0),
    ("12",  "Up-Conditioning_VNS/HRPilot-27/UPVNS13_HRPILOT-27_BOOTH4_100US_10KHZ_11-5-26",        10000.0),
    ("13",  "Up-Conditioning_VNS/HRPilot-27/UPVNS14_HRPILOT-27_BOOTH4_100US_10KHZ_11-6-26",        10000.0),
    ("14",  "Up-Conditioning_VNS/HRPilot-27/UPVNS15_HRPILOT-27_BOOTH4_100US_10KHZ_11-9-26",        10000.0),
   
   # HRPilot-31 Recordings Control
    #("-5",  "Control_Mode/HRPilot-31/CMN4_HRPILOT-31_BOOTH1_250US_10KHZ_10-28-26",        10000.0),
    #("-4",  "Control_Mode/HRPilot-31/CMN5_HRPILOT-31_BOOTH1_250US_10KHZ_10-29-26",        10000.0),
    #("-3",  "Control_Mode/HRPilot-31/CMN6_HRPILOT-31_BOOTH1_250US_10KHZ_10-29-26",        10000.0),
    #("-2",  "Control_Mode/HRPilot-31/CMN7_HRPILOT-31_BOOTH1_250US_10KHZ_10-30-26",        10000.0),
    #("-1",  "Control_Mode/HRPilot-31/CMN7_REAL_HRPILOT-31_BOOTH1_250US_10KHZ_10-30-26",        10000.0),
    
    
    # HRPilot-33 Recordings
    #("Calib1 HRPILOT-33 250US",  "Calibration/HRPilot-33/CALIB1_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
    #("Calib1PT2 HRPILOT-33 250US",  "Calibration/HRPilot-33/CALIB1_PT2_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
    #("CM1 HRPILOT-33 250US",  "Calibration/HRPilot-33/CM1_FILTFILT_HRPILOT-33_BOOTH3_250US_10KHZ_9-28-26",        10000.0),
    #("CM2 HRPILOT-33 250US",  "Calibration/HRPilot-33/CM2_HRPILOT-33_BOOTH3_100US_10KHZ_9-30-26",        10000.0),
    #("CM2 FT HRPILOT-33 250US",  "Calibration/HRPilot-33/CM2_HRPILOT-33_BOOTH3_250US_10KHZ_9-30-26",        10000.0),
    
    
    
    # HRPilot-34 Recordings
    #("Calib4 HRPILOT-34 100US",  "Calibration/HRPilot-34/CALIB4_FILTFILT_HRPILOT-34_BOOTH1_250US_10KHZ_9-23-26",        10000.0),
    #("CM1 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM1_FILTFILT_HRPILOT-34_BOOTH1_250US_10KHZ_9-24-26",        10000.0),
    #("CM2 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM2_FILTFILT_FIXED_HRPILOT-34_BOOTH2_250US_10KHZ_9-25-26",        10000.0),
    #("CM3 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM3_HRPILOT-34_BOOTH1_250US_10KHZ_9-28-26",        10000.0),
    #("CM4 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM4_HRPILOT-34_BOOTH1_250US_10KHZ_9-30-26",        10000.0),
    #("CM5 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM5_HRPILOT-34_BOOTH1_250US_10KHZ_10-1-26",        10000.0),
    #("CM6 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM6_HRPILOT-34_BOOTH1_250US_10KHZ_10-2-26",        10000.0),
    #("CM7 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM7_HRPILOT-34_BOOTH1_250US_10KHZ_10-5-26",        10000.0),
    #("CM8 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM8_HRPILOT-34_BOOTH3_250US_10KHZ_10-6-26",        10000.0),
    #("CM9 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM9_HRPILOT-34_BOOTH3_250US_10KHZ_10-7-26",        10000.0),
    #("CM10 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM10_HRPILOT-34_BOOTH3_250US_10KHZ_10-8-26",        10000.0),
        
    
    
    
    # HRPilot-36 Recordings
    #("Calib2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CALIB2_HRPILOT-36_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    #("CM1 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM1_HRPILOT-36_BOOTH2_250US_10KHZ_9-14-26",        10000.0),
    #("CM2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM2_HRPILOT-36_BOOTH2_250US_10KHZ_9-15-26",        10000.0),
    #("CM3 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
    #("CM3 FT OFfline HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_FT_OFFLINE_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
    #("CM4 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26",        10000.0),
    #("CM5 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM5_HRPILOT-36_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
    #("CM6 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM6_HRPILOT-36_BOOTH1_250US_10KHZ_9-22-26",        10000.0),
    #("CM7 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM7_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-23-26",        10000.0),
    #("CM8 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM8_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-24-26",        10000.0),
    #("CM9 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM9_FILTFILT_FIXED_HRPILOT-36_BOOTH1_250US_10KHZ_9-25-26",        10000.0),
    #("CM10 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM10_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-28-26",        10000.0),
    #("CM11 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM11_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-29-26",        10000.0),
    #("CM12 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM12_HRPILOT-36_BOOTH2_250US_10KHZ_9-30-26",        10000.0),
    #("CM13 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM13_HRPILOT-36_BOOTH2_250US_10KHZ_10-1-26",        10000.0),
    
    
    
    # HRPilot-36 Up-VNS Conditioning Recordings

    #("-5",  "Control_Mode/HRPilot-36/UPVNS3_HRPILOT-36_BOOTH2_250US_10KHZ_10-6-26",        10000.0),
    #("-4",  "Control_Mode/HRPilot-36/UPVNS5_HRPILOT-36_BOOTH2_250US_10KHZ_10-8-26",        10000.0),
    #("-3",  "Control_Mode/HRPilot-36/UPVNS6_HRPILOT-36_BOOTH2_250US_10KHZ_10-9-26",        10000.0),
    #("-2",  "Control_Mode/HRPilot-36/UPVNS7_HRPILOT-36_BOOTH2_250US_10KHZ_10-12-26",        10000.0),
    #("-1",  "Control_Mode/HRPilot-36/UPVNS8_HRPILOT-36_BOOTH2_250US_10KHZ_10-13-26",        10000.0),
    #("1",  "Up-Conditioning_VNS/HRPilot-36/UPVNS9_HRPILOT-36_BOOTH2_250US_10KHZ_10-15-26",        10000.0),
    #("2",  "Up-Conditioning_VNS/HRPilot-36/UPVNS10_NEW_SETUP_HRPILOT-36_BOOTH2_250US_10KHZ_10-16-26",        10000.0),
    #("3",  "Up-Conditioning_VNS/HRPilot-36/UPVNS11_NEW_SET-UP_HRPILOT-36_BOOTH2_250US_10KHZ_10-17-26",        10000.0),
    #("4",  "Up-Conditioning_VNS/HRPilot-36/UPVNS12_HRPILOT-36_BOOTH2_250US_10KHZ_10-19-26",        10000.0),
    #("5",  "Up-Conditioning_VNS/HRPilot-36/UPVNS13_HRPILOT-36_BOOTH2_250US_10KHZ_10-20-26",        10000.0),
    

    
    
    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

8 recording(s) configured.
  [0] '7'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS8_HRPILOT-27_BOOTH4_100US_10KHZ_10-30-26  (sample_rate=10000.0 Hz)
  [1] '8'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS9_HRPILOT-27_BOOTH4_100US_10KHZ_10-31-26  (sample_rate=10000.0 Hz)
  [2] '9'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS10_HRPILOT-27_BOOTH4_100US_10KHZ_11-2-26  (sample_rate=10000.0 Hz)
  [3] '10'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS11_HRPILOT-27_BOOTH4_100US_10KHZ_11-3-26  (sample_rate=10000.0 Hz)
  [4] '11'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS12_HRPILOT-27_BOOTH4_100US_10KHZ_11-4-26  (sample_rate=10000.0 Hz)
  [5] '12'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS13_HRPILOT-27_BOOTH4_100US_10KHZ_11-5-26  (sample_rate=10000.0 Hz)
  [6] '13'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS14_HRPILOT-27_BOOTH4_100US_10KHZ_11-6-26  (sample_rate=10000.0 Hz)
  [7] '14'  →  Up-Conditioning_VNS/HRPilot-27/UPVNS15_HRPILOT-27_BOOTH4_100US_10KHZ_11-9-26  (sample_rate=10000.0 Hz)


In [3]:
# ── Load all recordings (auto-detects V2 / V3) ──────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

# Saved analyses to load instead of / as well as the binary files: one path or a list
# of .pkl files written by "Save Analysis to Disk" (Section 6d, INCLUDE_RECORDINGS = True).
LOAD_ANALYSIS_PICKLE = "Up-Conditioning_VNS\HRPilot-27\HRPilot-27_h_reflex_analysis.pkl"   # e.g. "h_reflex_analysis_20261028_131500.pkl" or [path1, path2]
# True  = pickle recordings first, then every folder in RECORDING_DIRS (e.g. a new .hrs6)
# False = pickle recordings only (RECORDING_DIRS above is ignored)
ALSO_LOAD_RECORDING_DIRS = True

saved_analysis, _sets = None, []
_pkls = ([LOAD_ANALYSIS_PICKLE] if isinstance(LOAD_ANALYSIS_PICKLE, str)
         else list(LOAD_ANALYSIS_PICKLE or []))
for _p in _pkls:
    saved_analysis = load_analysis_pickle(_p)   # the last one is kept for its DataFrames / config
    if not saved_analysis.get('recordings_raw'):
        raise ValueError(f'{_p} has no recordings (saved with INCLUDE_RECORDINGS = False '
                         f'or by the old save cell) — use load_analysis_pickle() for its DataFrames.')
    _sets.append((saved_analysis['recordings_raw'], saved_analysis['config'].get('RECORDING_DIRS') or []))
if not _pkls or ALSO_LOAD_RECORDING_DIRS:
    _sets.append((load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM),
                  RECORDING_DIRS))
_all_recordings_raw, RECORDING_DIRS = merge_recording_sets(*_sets)
# Analysis cells use _all_recordings; the Trial Exclusion section below
# replaces it with a cleaned copy. _all_recordings_raw is never modified.
_all_recordings = _all_recordings_raw
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded: {list(_all_recordings)}.  '
      f'Active: {_active_rec_label!r}')

<>:7: SyntaxWarning: invalid escape sequence '\H'
<>:7: SyntaxWarning: invalid escape sequence '\H'
C:\Users\dal866445\AppData\Local\Temp\ipykernel_34044\1775010890.py:7: SyntaxWarning: invalid escape sequence '\H'
  LOAD_ANALYSIS_PICKLE = "Up-Conditioning_VNS\HRPilot-27\HRPilot-27_h_reflex_analysis.pkl"   # e.g. "h_reflex_analysis_20261028_131500.pkl" or [path1, path2]


Loaded Up-Conditioning_VNS\HRPilot-27\HRPilot-27_h_reflex_analysis.pkl  (format 1, saved 2026-10-31T14:06:52)
  contents: ['config', 'trials_df', 'xr_df', 'xr_summary', 'settings_df', 'excluded_trials_df', 'xr_cache', 'xr_pool_cache', 'recordings_raw', 'recordings']
  recordings: ['-5', '-4', '-3', '-2', '-1', '1', '2', '3', '4', '5', '6']  (no raw EMG blocks)

── Loading: '7'  (Up-Conditioning_VNS/HRPilot-27/UPVNS8_HRPILOT-27_BOOTH4_100US_10KHZ_10-30-26)
   .hrs1: not found
   .hrs2: not found
[_make_v3_block_loop] Falling back to byte-pattern scan for block_id=11 …
[_make_v3_block_loop] Found 1329 additional trial position(s)
   .hrs6: 1330 trials  (Up Condition VNS)
   [OFFLINE] 7/Up Condition VNS (.hrs6): 1322/1330 trials rebuilt (raw→diff→filtfilt ±200 ms; matched zero_phase)  (8 kept stored — alignment miss)
   App V3  |  Stages: ['up_cond_vns']  |  SR: 10000.0 Hz
   Settings sidecars: 1/1 stage(s)

── Loading: '8'  (Up-Conditioning_VNS/HRPilot-27/UPVNS9_HRPILOT-27_BOOTH4_100US_1

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [4]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: '-5'
  Control Mode (.hrs2): 1032 trials
  Frequency Test (.hrft): 131 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [5]:
# ── Recording & Stage Viewer Factory ───────────────────────────────────────────
# Each viewer section below has its own independent Recording + Stage dropdowns.
# Set each viewer to a different recording/stage to compare them side by side.
# make_viewer() is defined in helpers.py
from IPython.display import display as _disp

# ── Loaded recordings summary ────────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()
print('Each viewer below has its own Recording + Stage dropdowns for independent selection.')

# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()

19 recording(s) loaded:
  '-5'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 1032 trials
    · Frequency Test (.hrft): 131 trials
  '-4'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 1200 trials
  '-3'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 887 trials
    · Frequency Test (.hrft): 134 trials
  '-2'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 805 trials
  '-1'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 818 trials
  '1'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 957 trials
  '2'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 1123 trials
  '3'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 977 trials
  '4'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 953 trials
  '5'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 1039 trials
  '6'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 1071 trials
  '7'  (App V3  |  10000.0 Hz)
    · Up Condition VNS (.hrs6): 1330 trials
  '8'  (App V3 

In [6]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_tl(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Trial Timeline: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_actual_trial_timeline(trials, header=header)

_tl_widget, _tl_render = make_viewer(_all_recordings, _active_rec_label, _render_tl)
_disp(_tl_widget)
_tl_render()

# Section 3b: "Most Recent Background" + "Background EMG Level"

Recreates the H-Reflex App recruitment-curve trial-plot widgets from `MhRecruitmentCurveStage.get_trial_plot_options`:

- **Most recent background** bar chart of the pre-stim |EMG| bins.
- **EMG Level** scatter of background grand means across trials.

Bins are reconstructed from `hrs2_emg_blocks` over a fixed monitoring window (default 2500 ms ending at trigger time).

In [7]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_bg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Background EMG: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_background_emg_views(trials, emg_blocks, monitoring_window_ms=2500)

_bg_widget, _bg_render = make_viewer(_all_recordings, _active_rec_label, _render_bg)
_disp(_bg_widget)
_bg_render()

# Section 6: HRS2 Detailed Analysis

Interactive averaged-waveform paged grid (with M/H peak markers and signal overlays) plus the normalized and raw recruitment curves. The cell below sets the analysis parameters; the cell after that calls `plot_hrs2_analysis`.

In [30]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
#M_WAVE_START_MS = 3
#M_WAVE_END_MS= 5.5
#H_WAVE_START_MS = 9.0
#H_WAVE_END_MS   = 12

M_WAVE_START_MS = 2 
M_WAVE_END_MS= 4
H_WAVE_START_MS = 6   
H_WAVE_END_MS   = 10

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [31]:
# ── Failed / Alignment-Miss Trial Viewer ──────────────────────────────────────
# Shows trials where offline xcorr reconstruction failed (stored app-filtered data kept).
# Controls mirror make_ft_viewer: Y auto/min/max, Fig W/H, Pre/Post ms.
# Subplots: Show Stim ADC, Show Sync (analog), Show Sync DIG IN (step-function TTL).
# Toggle "Show ±10 s context" for the wider raw EMG-block context figure.
#
# Red dashed line = stored onset (onset_sample_index)
# Orange dotted line = stored stim end (stim_end_sample_index), if in view
from IPython.display import display as _disp
_failed_viewer = make_failed_trials_viewer(_all_recordings_raw,
                                            pre_ms=10, post_ms=50)
_disp(_failed_viewer)

# Section 2c: Trial Exclusion & Pre-Analysis Filter

Runs **before** any analysis or DataFrame building. Two optional steps, applied in order:

1. **Failed-trial exclusion.** Removes the trials flagged by the Failed Trial Viewer above (default: its *All offline* group).
2. **Metric pre-filter.** Keeps only trials whose M-wave / H-wave / H:M ratio / background is within a target ± tolerance.

The apply cell reassigns `_all_recordings`, so **every cell below uses the cleaned data**. The untouched load stays in `_all_recordings_raw`, which the Failed Trial Viewer reads. Re-running is safe because the apply cell always starts from `_all_recordings_raw`.

Outputs: `excluded_trials_df` (one row per removed failed trial, `trial_idx` = the viewer's `trial [i]`) and `prefilter_metrics_df` (per-trial metrics + `kept`).

In [32]:
# ── Trial Exclusion & Pre-Filter Configuration ────────────────────────────────
# 1) Failed-trial exclusion
EXCLUDE_FAILED_TRIALS = True
# 'all' = the Failed Trial Viewer's "All offline" group: no sync, alignment miss,
# recalculated bad stim ADC, frozen stim ADC / sync (offline), onset mismatch.
# Or list keys, e.g. ['no_sync', 'recalc_stim', 'align_miss'].
# App flags ('app_stim', 'app_sync', 'app_both', 'app_vns') are opt-in only —
# the app's live frozen-run check over-flags clean trials.
EXCLUDE_CATEGORIES = 'all'
EXCLUDE_STAGES     = None     # None = all stages, or e.g. ['control_mode']

# 2) Metric pre-filter (applied after failed-trial exclusion)
PREFILTER_ENABLED = False
# Metric to filter on: 'M_WAVE' | 'H_WAVE' | 'HM_RATIO' | 'BACKGROUND'
PREFILTER_METRIC        = 'M_WAVE'
PREFILTER_TARGET        = 1000      # µV  (or dimensionless ratio for HM_RATIO)
PREFILTER_TOLERANCE_PCT = 50      # ±%  around target

PREFILTER_LO = PREFILTER_TARGET * (1 - PREFILTER_TOLERANCE_PCT / 100)
PREFILTER_HI = PREFILTER_TARGET * (1 + PREFILTER_TOLERANCE_PCT / 100)
# Uncomment to override with manual bounds instead:
# PREFILTER_LO = 100.0
# PREFILTER_HI = 200.0
PREFILTER_STAGES = None       # None = all stages, or e.g. ['control_mode']

print(f"Failed-trial exclusion : {'ON' if EXCLUDE_FAILED_TRIALS else 'OFF'}"
      f"  (categories={EXCLUDE_CATEGORIES!r}, stages={EXCLUDE_STAGES or 'all'})")
print(f"Metric pre-filter      : {'ON' if PREFILTER_ENABLED else 'OFF'}")
if PREFILTER_ENABLED:
    print(f"  Metric  : {PREFILTER_METRIC}")
    print(f"  Target  : {PREFILTER_TARGET}  ±{PREFILTER_TOLERANCE_PCT}%")
    print(f"  Bounds  : [{PREFILTER_LO:.2f},  {PREFILTER_HI:.2f}]")
    print(f"  Stages  : {PREFILTER_STAGES or 'all'}")

Failed-trial exclusion : ON  (categories='all', stages=all)
Metric pre-filter      : OFF


In [33]:
# ── Apply trial exclusion + pre-filter ────────────────────────────────────────
# Always starts from the raw load, so re-running with new settings is safe.
# Uses PRE_AVG_MS / POST_AVG_MS and the M/H window constants from the Configuration cell.
_all_recordings = _all_recordings_raw
excluded_trials_df   = None
prefilter_metrics_df = None

if EXCLUDE_FAILED_TRIALS:
    _all_recordings, excluded_trials_df = exclude_failed_trials(
        _all_recordings, categories=EXCLUDE_CATEGORIES, stages=EXCLUDE_STAGES)
else:
    print('Failed-trial exclusion OFF — failed trials kept.')
print()

if PREFILTER_ENABLED:
    _all_recordings, prefilter_metrics_df = filter_recordings_by_metric(
        _all_recordings, PREFILTER_METRIC, PREFILTER_LO, PREFILTER_HI,
        PRE_AVG_MS, POST_AVG_MS,
        M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, H_WAVE_END_MS,
        stages=PREFILTER_STAGES)
else:
    print('Metric pre-filter OFF.')

print('\nAll cells below use the cleaned _all_recordings '
      '(raw data kept in _all_recordings_raw).')
excluded_trials_df

Failed-trial exclusion  (categories: all offline)
  -5  [Control Mode (.hrs2)]:  kept 1004/1032  (removed 28)
  -5  [Frequency Test (.hrft)]:  kept 107/131  (removed 24)
  -4  [Control Mode (.hrs2)]:  kept 1174/1200  (removed 26)
  -3  [Control Mode (.hrs2)]:  kept 880/887  (removed 7)
  -3  [Frequency Test (.hrft)]:  kept 109/134  (removed 25)
  -2  [Control Mode (.hrs2)]:  kept 801/805  (removed 4)
  -1  [Control Mode (.hrs2)]:  kept 817/818  (removed 1)
  1  [Up Condition VNS (.hrs6)]:  kept 937/957  (removed 20)
  2  [Up Condition VNS (.hrs6)]:  kept 1096/1123  (removed 27)
  3  [Up Condition VNS (.hrs6)]:  kept 954/977  (removed 23)
  4  [Up Condition VNS (.hrs6)]:  kept 952/953  (removed 1)
  5  [Up Condition VNS (.hrs6)]:  kept 1030/1039  (removed 9)
  6  [Up Condition VNS (.hrs6)]:  kept 1064/1071  (removed 7)
  7  [Up Condition VNS (.hrs6)]:  kept 1322/1330  (removed 8)
  8  [Up Condition VNS (.hrs6)]:  kept 1026/1111  (removed 85)
  9  [Up Condition VNS (.hrs6)]:  kept 774/78

,recording,stage,trial_idx,amp_ma,categories,reasons
0,-5,Control Mode (.hrs2),0,0.170,align_miss,ALIGNMENT MISS — stored data
1,-5,Control Mode (.hrs2),7,0.171,align_miss,ALIGNMENT MISS — stored data
2,-5,Control Mode (.hrs2),354,0.157,align_miss,ALIGNMENT MISS — stored data
3,-5,Control Mode (.hrs2),444,0.151,align_miss,ALIGNMENT MISS — stored data
4,-5,Control Mode (.hrs2),460,0.167,align_miss,ALIGNMENT MISS — stored data
...,...,...,...,...,...,...
475,14,Up Condition VNS (.hrs6),1140,0.132,align_miss,ALIGNMENT MISS — stored data
476,14,Up Condition VNS (.hrs6),1240,0.138,align_miss,ALIGNMENT MISS — stored data
477,14,Up Condition VNS (.hrs6),1275,0.135,align_miss,ALIGNMENT MISS — stored data
478,14,Up Condition VNS (.hrs6),1295,0.135,align_miss,ALIGNMENT MISS — stored data


In [34]:
# ── Pre-compute Comparison Data ───────────────────────────────────────────────────────
# Computed once here (after configuration constants are set) for instant rendering
# in the comparison plot below. Re-run this cell if you change PRE_AVG_MS,
# POST_AVG_MS, M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, or H_WAVE_END_MS.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'Comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

# ── Pooled days ───────────────────────────────────────────────────────────────
# Used by "Pool days" in the comparison plot and by the Pooled HRS2 Analysis.
# Each entry: (pool label, (first label, last label)) — inclusive range in
# RECORDING_DIRS order — or (pool label, ['label', 'label', ...]).
# Every trial of every member recording is pooled (each trial counts once).
from helpers import pool_recordings
XR_POOLS = [
    ('Week 0 (Pre)', ('-5', '-1')),
    ('Week 1 (VNS)',   ('1', '5')),
]
# Bar color per pool when "Pool days" is on (recordings in no pool: light gray).
XR_POOL_COLORS = {'Week 0 (Pre)': 'tab:blue', 'Week 1 (VNS)': 'tab:red'}
_xr_pool_recs, _xr_pool_dirs = pool_recordings(_all_recordings, RECORDING_DIRS, XR_POOLS)
_xr_pool_cache = {**_xr_cache, **compute_h_comparison_data(
    _xr_pool_recs, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)}


Comparison data pre-computed for 19 recording(s), 3 stage(s): ['control_mode', 'frequency_test', 'up_cond_vns']
'Week 0 (Pre)': ['-5', '-4', '-3', '-2', '-1']
    Control Mode (.hrs2): 4676 trials  (-5=1004, -4=1174, -3=880, -2=801, -1=817)
  Note: 'Week 1 (VNS)' / Up Condition VNS (.hrs6): Delay Sync DIGIN channel differs between members {'1': None, '2': None, '3': 4, '4': 4, '5': 4} — pooled plots use the first one.
'Week 1 (VNS)': ['1', '2', '3', '4', '5']
    Up Condition VNS (.hrs6): 4969 trials  (1=937, 2=1096, 3=954, 4=952, 5=1030)


In [35]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = True   # True → collapse every amplitude into one group
MERGED_GROUPS = []
#[(0,0.108), (0.108,0.23), (0.23,0.43), (0.43,0.50), (0.50,0.60), (0.60,0.70)]

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = True  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [ ]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ──────────────────────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _sr = sr or h1h.sample_rate
    tp = _apply_merge(trials)
    print(f'\n── Analysis: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_analysis(
        tp, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=_sr,
        emg_blocks=emg_blocks,
    )

_ana_widget, _ana_render = make_viewer(
    _all_recordings, _active_rec_label, _render_ana,
    stage_filter=lambda sk, *_a: sk != 'frequency_test',
)
_disp(_ana_widget)
_ana_render()


# Section 6a: Pooled HRS2 Analysis (Days Collapsed)

Same averaged-waveform grid + recruitment curve as **HRS2 Analysis** above, but each **Pool** collapses several recordings into one: every trial of every member recording (after failed-trial exclusion / pre-filter) is averaged together, so each trial counts once.

Pools are set by `XR_POOLS` in the **Pre-compute Comparison Data** cell (default: days −5…−1 and days 1…5). Merge settings (`MERGE_ALL` / `MERGED_GROUPS`) apply exactly as in HRS2 Analysis.

In [37]:
# ── Pooled HRS2 Analysis: one averaged waveform per pool of days ──────────────
# Pools come from XR_POOLS (Pre-compute Comparison Data cell). Reuses _render_ana
# from the HRS2 Analysis cell, so the output format is identical.
from IPython.display import display as _disp

def _render_pool_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _pr = _xr_pool_recs[rec_label]
    _sk = next(k for k, v in _pr['stage_map'].items() if v[0] is trials)
    print('Pooled: ' + ', '.join(f'{m} ({n} trials)' for m, n in _pr['pool_counts'][_sk].items()))
    _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h)

if _xr_pool_recs:
    _pool_ana_widget, _pool_ana_render = make_viewer(
        _xr_pool_recs, next(iter(_xr_pool_recs)), _render_pool_ana,
        stage_filter=lambda sk, *_a: sk != 'frequency_test',
    )
    _disp(_pool_ana_widget)
    _pool_ana_render()
else:
    print('No pools — check XR_POOLS labels against RECORDING_DIRS.')


# M-Wave Stabilization Control Error 

Plots the M-wave stabilization controller output trial-by-trial.
- **Left axis** — `m_wave_error` (µV): difference between actual M-wave and the target set-point; falls back to `m_wave_window_median` when the controller was inactive.
- **Right axis** — `stimulation_amplitude_ma` (mA, orange): stimulation intensity actually used.


In [38]:
# ── M-Wave Control Error with Reference Lines ─────────────────────────────────
# Controls:
#   Auto checkbox  — reads target M-size from the recording's stored set-value
#   Target (µV)    — manual override (editable when Auto is unchecked)
#   ±Inner %       — tolerance band drawn as green dashed lines around the target
#   ±Outer %       — algorithm bounds drawn as red dotted lines (set to 0 to hide)
#   Update Plot    — re-render with current settings after changing parameters
from ipywidgets import Checkbox, FloatText, FloatSlider, Button, Output, VBox, HBox, Label
from IPython.display import display as _disp
import numpy as np

def _mw_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and any(
        not np.isnan(getattr(t, 'm_wave_error', float('nan'))) or
        not np.isnan(getattr(t, 'm_wave_window_median', float('nan')))
        for t in trials)

def _mw_auto_target(trials):
    vals  = [getattr(t, 'm_wave_set_value_uv', float('nan')) for t in trials]
    valid = [v for v in vals if not np.isnan(v)]
    return float(np.median(valid)) if valid else float('nan')

# ── Control widgets ────────────────────────────────────────────────────────────
_mw_auto_cb   = Checkbox(value=True, description='Auto (recording set-value)',
                          style={'description_width': 'initial'}, layout={'width': '270px'})
_mw_target_ft = FloatText(value=0.0, description='Target (µV):',
                           step=10, disabled=True, layout={'width': '190px'})
_mw_inner_sl  = FloatSlider(value=25.0, min=0, max=100, step=1,
                              description='±Inner %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_outer_sl  = FloatSlider(value=50.0, min=0, max=200, step=1,
                              description='±Outer %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_apply_btn = Button(description='Update Plot', button_style='primary',
                        layout={'width': '120px'})
_mw_info_lbl  = Label(value='')

_mw_controls = VBox([
    HBox([_mw_auto_cb, _mw_target_ft, _mw_info_lbl]),
    HBox([_mw_inner_sl, _mw_outer_sl, _mw_apply_btn]),
])

def _render_mw(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    if _mw_auto_cb.value:
        _auto = _mw_auto_target(trials)
        if not np.isnan(_auto):
            _mw_target_ft.value = _auto
            _mw_info_lbl.value  = f'Auto: {_auto:.1f} µV'
        else:
            _mw_info_lbl.value  = 'Auto: no set-value stored'
        _target = None
    else:
        _target = _mw_target_ft.value
        _mw_info_lbl.value = f'Override: {_target:.1f} µV'

    print(f'\n── M-Wave Control Error: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_mwave_control_error(
        trials, header,
        sample_rate=sr or h1h.sample_rate,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        target_uv=_target,
        inner_pct=_mw_inner_sl.value,
        outer_pct=_mw_outer_sl.value,
    )

_has_any_mw = any(
    any(_mw_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)

if _has_any_mw:
    _mw_widget, _mw_render = make_viewer(
        _all_recordings, _active_rec_label, _render_mw,
        stage_filter=_mw_stage_filter,
    )

    def _mw_auto_toggle(change):
        _mw_target_ft.disabled = change['new']
        _mw_render()

    _mw_auto_cb.observe(_mw_auto_toggle, names='value')
    _mw_apply_btn.on_click(lambda b: _mw_render())

    _disp(_mw_controls)
    _disp(_mw_widget)
    _mw_render()
else:
    print("No M-wave stabilization data available in any loaded recording "
          "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")

# M-Wave Stabilization: Grouped by Distribution Window Size

Bar + whisker + per-trial-dots plot for recordings using the M-wave stabilization algorithm. Groups are detected from the per-trial stored `m_wave_distribution_window_size` (so a mid-recording change to that setting automatically shows up as its own group).

- **Bar height** = group mean
- **Whisker** = ±1 SD within the group
- **Dots** = every individual trial's own value

**Metric** toggle:
- **M-wave Size** (default) — each trial's own M-wave size (µV)
- **|Size − Set Value|** — `abs(m_wave_size - m_wave_set_value_uv)`: how far that trial's own M-wave size landed from the stabilizer's target
- **H-wave Size (offline)** — each trial's H-wave size (µV) recomputed from `trial_data` over `H_WAVE_START_MS`–`H_WAVE_END_MS`, minus the −55 to −5 ms pre-stim background
- **H-wave Size (in-session)** — the app's live `h_wave_response` (µV), i.e. the value the reward decision was made on

**Dots** are grey; on conditioning stages (.hrs3–.hrs6) trials that received the reward (`vns_delivered` / `pellet_delivered`) are drawn **red**, and each group's label shows how many.

**Note (Control Mode .hrs2 only):** `m_wave_distribution_window_size` isn't readable from the current file format yet (verified directly against real bytes — the already-known "opaque trailer" after the M-wave block doesn't contain it, it re-echoes `onset_sample_index`/`stim_end_sample_index` instead). Until the real byte position is found, every trial reports the same placeholder window size and this shows as a single group — the bar/whisker/dots and metric toggle are fully working today, just not yet split by window size.

In [39]:
# ── M-Wave Stabilization Groups ─────────────────────────────────────────────
# "Downsample to smallest group": when there are 2+ groups, every larger group is
# reduced to the smallest group's n (Fixed Intensity counts as a group).
#   Method  random — seeded draw without replacement (same seed → same subset)
#           even   — evenly spaced through the session
#           first  — earliest n trials
# Per-group n, mean, SD, CV (= SD/|mean|×100), median/IQR and stim amplitude are
# printed as tab-separated text below the plot — select + copy, pastes into Excel/Sheets.
# Dots are grey; trials that got the conditioning reward (VNS / pellet) are red.
from ipywidgets import ToggleButtons, Checkbox, Dropdown, IntText, VBox, HBox, Output
from IPython.display import display as _disp
from helpers import format_mwave_stabilization_summary
import numpy as np

def _mws_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and any(
        not np.isnan(getattr(t, 'm_wave_error', float('nan'))) or
        not np.isnan(getattr(t, 'm_wave_window_median', float('nan')))
        for t in trials)

_mws_metric_tog = ToggleButtons(
    options=[('M-wave Size', 'm_wave_size'), ('|Size − Set Value|', 'abs_error'),
             ('H-wave Size (offline)', 'h_wave_size'), ('H-wave Size (in-session)', 'h_wave_app')],
    value='m_wave_size', description='Metric:',
)
_mws_ds_chk    = Checkbox(value=False, description='Downsample to smallest group',
                          indent=False, layout={'width': '230px'})
_mws_ds_method = Dropdown(options=['random', 'even', 'first'], value='random',
                          description='Method:', layout={'width': '180px'}, disabled=True)
_mws_ds_seed   = IntText(value=0, description='Seed:', layout={'width': '140px'}, disabled=True)

def _mws_on_ds(change):
    _mws_ds_method.disabled = not _mws_ds_chk.value
    _mws_ds_seed.disabled   = not _mws_ds_chk.value or _mws_ds_method.value != 'random'
    _mws_render()

_has_any_mws = any(
    any(_mws_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)

if _has_any_mws:
    def _render_mws(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
        print(f'\n── M-Wave Stabilization Groups: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
        _mws_summary = plot_mwave_stabilization_groups(
            trials, header, sr or h1h.sample_rate,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
            metric=_mws_metric_tog.value,
            downsample=_mws_ds_chk.value,
            downsample_method=_mws_ds_method.value,
            downsample_seed=_mws_ds_seed.value,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        )
        if _mws_summary:
            print(format_mwave_stabilization_summary(_mws_summary))

    _mws_widget, _mws_render = make_viewer(
        _all_recordings, _active_rec_label, _render_mws,
        stage_filter=_mws_stage_filter,
    )
    _mws_metric_tog.observe(lambda c: _mws_render(), names='value')
    for _w in (_mws_ds_chk, _mws_ds_method, _mws_ds_seed):
        _w.observe(_mws_on_ds, names='value')
    _disp(VBox([_mws_metric_tog,
                HBox([_mws_ds_chk, _mws_ds_method, _mws_ds_seed]),
                _mws_widget]))
    _mws_render()
else:
    print("No M-wave stabilization data available in any loaded recording "
          "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")

# H-wave Size per Trial: Conditioning Reward Timeline

For conditioning stages (.hrs3–.hrs6). One dot per trial, H-wave size on the y-axis and trial number on the x-axis.

- **Grey dots** — no reward; **red dots** — reward delivered (`vns_delivered` / `pellet_delivered`, as saved by the app). This flag means the app *commanded* the reward TTL; the pulse fires after the trial's recording window closes, so it isn't in `trial_data` / `vns_adc_data`.
- **Green dotted line** — the success threshold the app applied to each trial (70th percentile of its rolling 100-trial in-session H-wave window). The app saves each trial's threshold *before* adding that trial's own H-wave, then decides with the updated value, so the line uses the value saved on the *next* trial — this reproduces every saved reward decision.
- **H-wave** toggle — **In-session (app)**: the app's live `h_wave_response` (the value the decision used). **Offline window**: recomputed from `trial_data` over `H_WAVE_START_MS`–`H_WAVE_END_MS`. The threshold line is always the app's, so in Offline mode it only lines up exactly when your offline H window matches the app's (see the `.settings.json` H window).
- **Scope** toggle — **Selected recording**, or **All recordings**: every loaded recording that has the selected stage, concatenated in `RECORDING_DIRS` order with a dashed divider between sessions. The threshold restarts each session because the app rebuilds its window at every session start.

Per-session trial counts, reward counts and means print below the plot as tab-separated text (select + copy, pastes into Excel/Sheets).


In [40]:
# ── H-wave Size per Trial: Conditioning Reward Timeline ────────────────────────
# Grey = no reward, red = reward delivered, green dotted = success threshold the
# app applied to that trial. See the markdown above for the toggles.
from ipywidgets import ToggleButtons, VBox, HBox, HTML, Checkbox
from IPython.display import display as _disp
from helpers import plot_h_wave_reward_timeline, format_h_wave_reward_summary, get_trial_reward

def _hrt_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and get_trial_reward(trials[0])[1] is not None

_hrt_src_tog = ToggleButtons(
    options=[('In-session (app)', 'in_session'), ('Offline window', 'offline')],
    value='in_session', description='H-wave:',
)
_hrt_scope_tog = ToggleButtons(
    options=[('Selected recording', 'one'), ('All recordings', 'all')],
    value='one', description='Scope:',
)
# Display: Compact = figure height x0.7; Large text = big bold poster text (default on)
_hrt_compact_cb = Checkbox(value=False, description='Compact', indent=False,
                           layout={'width': '100px'})
_hrt_large_cb = Checkbox(value=True, description='Large text', indent=False,
                         layout={'width': '110px'})
_hrt_display_row = HBox([HTML('<b style="line-height:32px">Display:</b>&nbsp;'),
                         _hrt_compact_cb, _hrt_large_cb])

_has_any_hrt = any(
    any(_hrt_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)

if _has_any_hrt:
    def _render_hrt(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
        if _hrt_scope_tog.value == 'all':
            _sessions = [(_rl, _t, _h, _rd['sample_rate'] or _rd['hrs1_header'].sample_rate)
                         for _rl, _rd in _all_recordings.items()
                         for _sk, (_t, _h, _e, _lbl) in _rd['stage_map'].items()
                         if _lbl == stage_label and _t]
        else:
            _sessions = [(rec_label, trials, header, sr or h1h.sample_rate)]
        print(f'\n── H-wave Reward Timeline: {stage_label}  '
              f'({len(_sessions)} session(s))  [{", ".join(s[0] for s in _sessions)}]')
        _hrt_summary = plot_h_wave_reward_timeline(
            _sessions, source=_hrt_src_tog.value,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
            poster=_hrt_large_cb.value, compact=_hrt_compact_cb.value,
        )
        if _hrt_summary:
            print(format_h_wave_reward_summary(_hrt_summary, get_trial_reward(_sessions[0][1][0])[1]))

    _hrt_widget, _hrt_render = make_viewer(
        _all_recordings, _active_rec_label, _render_hrt,
        stage_filter=_hrt_stage_filter,
    )
    for _w in (_hrt_src_tog, _hrt_scope_tog, _hrt_compact_cb, _hrt_large_cb):
        _w.observe(lambda c: _hrt_render(), names='value')
    _disp(VBox([_hrt_src_tog, _hrt_scope_tog, _hrt_display_row, _hrt_widget]))
    _hrt_render()
else:
    print("No conditioning-stage recordings (.hrs3–.hrs6) loaded.")


# Trial Viewer Grouping

Settings for the two per-trial viewers below (**VNS Trial Viewer** and **HRS2 Trial Viewer**).

- `TRIAL_GROUPS` / `TRIAL_GROUP_SIZE` add a **Trial group** dropdown that limits the grid to a set of trial numbers.
- **Trial Group Averages** (the cell right after the settings) collapses each trial group into one averaged waveform, overlays the groups on one plot and prints their M/H sizes.
- `USE_AMP_MERGE_IN_TRIAL_VIEWER` makes the **By stim intensity** dropdown use `MERGED_GROUPS` from the *Merged Amplitude Group* cell above. Trials are only filtered, so each panel still shows that trial's real amplitude.

Trial numbers are each trial's number in its session, counted before failed-trial exclusion and the pre-filter. These are the numbers shown in the panel titles.

In [41]:
# ── Trial Viewer Grouping (VNS + HRS2 Trial Viewers below) ─────────────────────
# Trial # = the trial's 1-based number in its session, counted BEFORE failed-trial
# exclusion / pre-filter (the number shown in each panel title).
#
# TRIAL_GROUPS accepts two styles (mix freely):
#   Range tuple   : (1, 50)          — trials where low <= trial # <= high
#   Explicit list : [3, 7, 12, 40]   — exactly those trial numbers
#
# Examples:
#   TRIAL_GROUPS = [(1, 50), (51, 100)]        ← first 50 vs next 50
#   TRIAL_GROUPS = [(1, 30), [45, 46, 90]]     ← range + explicit, mixed
#
# TRIAL_GROUP_SIZE = 50  → also add consecutive blocks 1–50, 51–100, … (None = off)
#
# USE_AMP_MERGE_IN_TRIAL_VIEWER = True → "By stim intensity" lists MERGED_GROUPS
# (from the Merged Amplitude Group cell above) as single entries. MERGE_ALL is
# ignored here: the "All trials" view already shows every amplitude.
# Groups combine: pick a Trial group, then "By stim intensity" filters within it.
TRIAL_GROUPS     = [(650, 750),(750,950)]
TRIAL_GROUP_SIZE = None
USE_AMP_MERGE_IN_TRIAL_VIEWER = True

def _trial_viewer_groups(trials, rec_label):
    """(trial_numbers, trial_groups, amp_groups) for plot_hrs2_trials."""
    nums = session_trial_numbers(trials, _all_recordings_raw.get(rec_label))
    groups = build_trial_number_groups(trials, TRIAL_GROUPS, nums,
                                       chunk_size=TRIAL_GROUP_SIZE)
    amp_groups = (_resolve_groups(trials, MERGED_GROUPS)
                  if USE_AMP_MERGE_IN_TRIAL_VIEWER and MERGED_GROUPS else None)
    return nums, groups, amp_groups

print("Trial groups:", TRIAL_GROUPS or "(none)", " | block size:", TRIAL_GROUP_SIZE)
print("Amp merge in trial viewers:",
      (MERGED_GROUPS or "(no MERGED_GROUPS)") if USE_AMP_MERGE_IN_TRIAL_VIEWER else "OFF")
print("Re-run the trial viewer cells or change Recording/Stage to apply.")


Trial groups: [(650, 750), (750, 950)]  | block size: None
Amp merge in trial viewers: (no MERGED_GROUPS)
Re-run the trial viewer cells or change Recording/Stage to apply.


In [42]:
# ── Trial Group Averages: one averaged waveform per trial group ───────────────
# Same panels as HRS2 Analysis, but each panel averages one group from
# TRIAL_GROUPS / TRIAL_GROUP_SIZE (Trial Viewer Grouping cell above), e.g. trials
# 500–700 vs 750–950, instead of one stim amplitude. Below the grid: every group
# average overlaid on one axis + an M/H size table (MRA − pre-stim BG).
# Uses each group's trials that survived failed-trial exclusion / pre-filter.
from IPython.display import display as _disp

def _render_grp_avg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _nums, _tgroups, _ = _trial_viewer_groups(trials, rec_label)
    print(f'\n── Trial Group Averages: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    if not _tgroups:
        print('No trial groups matched — set TRIAL_GROUPS / TRIAL_GROUP_SIZE in the '
              'Trial Viewer Grouping cell (trial #s are session numbers).')
        return
    plot_hrs2_analysis(
        trials, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
        trial_groups=_tgroups,
    )

_grp_avg_widget, _grp_avg_render = make_viewer(
    _all_recordings, _active_rec_label, _render_grp_avg,
    stage_filter=lambda sk, *_a: sk != 'frequency_test',
)
_disp(_grp_avg_widget)
_grp_avg_render()


# VNS Trial Viewer

The same per-trial grid + zoom viewer as the HRS2 Trial Viewer, showing **only the trials where VNS was delivered** (`vns_delivered`). Panel titles keep each trial's original session number.

Use the extra toggles to check whether the VNS was actually delivered. Each one adds its own row under the trial's EMG, on the same time axis:
- **Show VNS ADC** — the raw VNS ADC analog input (booth "VNS ADC" role, e.g. ZMQ ch 9), with a grey ±threshold band around its pre-trigger baseline and a dashed purple line at the detected pulse (with its latency from stim onset).
- **Show VNS Sync DIG IN** — the "VNS Sync" digital input (e.g. DIGIN 3) as a 0/1 step trace.

Widen **Post ms** in the Window box to see past the H-wave. The trial window runs to about +485 ms. Only recordings from the updated app (file_version ≥ 12) contain the pulse. On older files these rows stay flat, because the app fired VNS after the trial window had closed.


In [43]:
# ── VNS Trial Viewer: Interactive Per-Trial Grid + Zoom (VNS-delivered trials only) ──
# Same viewer as the HRS2 Trial Viewer, restricted to trials where the app delivered
# VNS (vns_delivered == 1). Panel titles keep each trial's original session number.
# Extra toggles (own axes, below the EMG, shared time axis):
#   Show VNS ADC         — raw VNS ADC analog input (booth "VNS ADC" role), ±threshold
#                          band around its pre-trigger baseline, dashed line at the pulse
#   Show VNS Sync DIG IN — the "VNS Sync" digital input as a 0/1 step trace
# The VNS rows have their own time axis spanning the whole stored VNS buffer
# (~585 ms after onset): EMG window shaded gold, reward decision = green dotted line.
from IPython.display import display as _disp

VNS_THRESHOLD_V = 0.5   # |VNS ADC − pre-trigger baseline| (V) that counts as the pulse

def _render_vns_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _vns_trials = [t for t in trials if getattr(t, 'vns_delivered', 0)]
    print(f'\n── VNS Trial Viewer: {stage_label}  '
          f'({len(_vns_trials)} VNS of {len(trials)} trials)  [{rec_label}]')
    if not _vns_trials:
        print('No VNS-delivered trials in this stage.')
        return
    # Per-trial viewer: no _apply_merge (it overwrites each trial's amplitude);
    # trial-number + merged-amp groups come from the Trial Viewer Grouping cell.
    tp = _vns_trials
    _nums, _tgroups, _agroups = _trial_viewer_groups(tp, rec_label)
    plot_hrs2_trials(
        tp, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
        trial_numbers=_nums,
        vns_threshold_v=VNS_THRESHOLD_V,
        trial_groups=_tgroups, amp_groups=_agroups,
    )

_vns_trv_widget, _vns_trv_render = make_viewer(
    _all_recordings, _active_rec_label, _render_vns_trv,
    stage_filter=lambda sk, trials, *_a: bool(trials) and hasattr(trials[0], 'vns_delivered'),
)
_disp(_vns_trv_widget)
_vns_trv_render()


In [44]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from IPython.display import display as _disp

def _render_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    # Per-trial viewer: no _apply_merge (it overwrites each trial's amplitude);
    # trial-number + merged-amp groups come from the Trial Viewer Grouping cell.
    tp = trials
    _nums, _tgroups, _agroups = _trial_viewer_groups(tp, rec_label)
    print(f'\n── Trial Viewer: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_trials(
        tp, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
        trial_numbers=_nums,
        trial_groups=_tgroups, amp_groups=_agroups,
    )

_trv_widget, _trv_render = make_viewer(
    _all_recordings, _active_rec_label, _render_trv,
    stage_filter=lambda sk, *_a: sk != 'frequency_test',
)
_disp(_trv_widget)
_trv_render()

# Section 6c: Frequency Test Analysis (V3 FT)

Three views available via the **View** toggle:

- **H/M MRA Per Pulse** — mean ± 1σ H-wave and M-wave MRA at each pulse position across all trials. Uses `pulse_h_wave_mra` / `pulse_m_wave_mra` pre-stored by the app (mean rectified average within each window). The most-recent trial is overlaid as a dashed line. Shows homosynaptic (rate-dependent) depression across the train.
- **Avg Waveforms** — paged 2×3 grid. Each tile is one pulse position: individual trial segments are drawn at low alpha, the bold black trace is the cross-trial mean. M-wave window is blue-shaded; H-wave window is green-shaded (matching the HRS2 Analysis viewer). MRA annotations are boxed above each window. A colorbar at the top encodes pulse # (blue = pulse 1, red = last). Use the **Page** slider to page through pulses.
- **H/M Peak Per Pulse** — same structure as H/M MRA Per Pulse, but computes `max(|EMG|)` within each wave window directly from the raw EMG trace, then averages across trials.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [45]:
# ── Frequency Test Analysis: Interactive Viewer ─────────────────────────────
from IPython.display import display as _disp
_disp(make_ft_viewer(
    _all_recordings, POST_PLOT_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
))

# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** is the **Mean Rectified Amplitude (MRA) of the averaged bipolar waveform** in the H-wave window, minus the MRA of the pre-stimulus background:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

Steps per amplitude group in each recording:
1. All trials at that amplitude are time-aligned and averaged → `avg_bip`
2. **H-wave MRA** = mean of |avg_bip| within [H_WAVE_START_MS, H_WAVE_END_MS]
3. **Background MRA** = mean of |avg_bip| in the pre-stimulus window (t < 0)
4. **H-reflex size** = H-wave MRA − background MRA

This matches the green `H: X.X µV` annotation shown in the HRS2 Analysis viewer above, with background subtracted.

The plot below shows one **box-and-whisker per recording** in `RECORDING_DIRS` order, with each amplitude group contributing one data point:
- **Box** — interquartile range (25th–75th percentile across amplitude groups)
- **Horizontal bar** — median
- **◆ diamond** — mean
- **Whiskers** — mean ± 1 standard deviation
- **n=** — number of amplitude groups in that recording for the selected stage
- **Dashed line** — connects per-recording means in `RECORDING_DIRS` order

In [ ]:
# ── H-Reflex Size Across Recordings: Comparison Plot ──────────────────────────────────────────────────────────
# Toggle between H-Reflex size, M-Wave size, and Background MRA (pre-stim EMG level).
# "Then stage" (optional): show a second stage's recordings to the right of the first,
#   in the same plot, separated by a dotted vertical line — e.g. Control Mode (.hrs2)
#   then Up Condition VNS (.hrs6).
# Optionally overlay a pooled mean +/- std ("Range average") computed across a
# contiguous range of recordings you pick below (drag either handle).
# The numbers behind the plot are printed as tab-separated text below it —
# select + copy, pastes into Excel/Sheets as columns.
# Re-run the pre-compute cell above if you change analysis configuration constants.
# "Compact" squeezes the x-axis so many recordings fit on screen at once (vertical
#   labels/text); "Box text" hides the n / amp / μ text above each box.
# "Large text" makes every label bold and much larger, with thicker lines and a bigger
#   figure — for posters / slides read from far away (box text switches to its short form).
# Metric "Stacked (all)" stacks H-Reflex, M-Wave and Background MRA vertically on one
#   shared recording axis (always compact; Compact checkbox ignored there).
# "Pool days" collapses each XR_POOLS group (e.g. days -5..-1, days 1..5) into one
#   bar (mean ± 1σ, colored by XR_POOL_COLORS) — every trial of its recordings pooled.
#   Set XR_POOLS / XR_POOL_COLORS in the pre-compute cell.
from ipywidgets import Dropdown, ToggleButtons, Checkbox, SelectionRangeSlider, Output, VBox, HBox, HTML
from helpers import format_h_comparison_summary, h_comparison_entry_labels, plot_h_reflex_comparison_stacked
from IPython.display import display as _disp

_xr_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'}
)
if _xr_stages:
    _xr_stage_d.value = next(iter(_xr_stages))

_xr_stage2_d = Dropdown(
    options=[('— none —', None)] + [(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    value=None, description='Then stage:', layout={'width': '480px'},
    style={'description_width': 'initial'},
)

_xr_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background'),
             ('Stacked (all)', 'all')],
    description='Metric:',
    style={'button_width': '160px'},
)

_xr_pool_chk = Checkbox(value=False, description='Pool days', indent=False,
                        layout={'width': '110px'}, disabled=not _xr_pool_recs)

def _xr_src():
    """(cache, recording order) — pooled per XR_POOLS when "Pool days" is on."""
    return ((_xr_pool_cache, _xr_pool_dirs) if _xr_pool_chk.value
            else (_xr_cache, RECORDING_DIRS))

def _xr_stage_sel():
    """Stage key, or [first, second] when a different 'Then stage' is chosen."""
    sk, sk2 = _xr_stage_d.value, _xr_stage2_d.value
    return [sk, sk2] if sk2 and sk2 != sk else sk

def _xr_entry_labels():
    if not _xr_stage_d.value:
        return ['']
    return h_comparison_entry_labels(*_xr_src(), _xr_stage_sel(), _xr_stages) or ['']

_xr_range_chk = Checkbox(value=False, description='', indent=False,
                         layout={'width': '28px'})
_xr_range_lbl = HTML("<span style='font-size:16px; font-weight:bold'>Show range average</span>")
_xr_range_sl = SelectionRangeSlider(
    options=_xr_entry_labels(),
    index=(0, len(_xr_entry_labels()) - 1),
    description='Avg range:',
    layout={'width': '560px'},
    disabled=True,
)

_xr_compact_chk = Checkbox(value=True, description='Compact', indent=False,
                          layout={'width': '110px'})
_xr_boxtxt_chk = Checkbox(value=True, description='Box text', indent=False,
                         layout={'width': '110px'})
_xr_large_chk = Checkbox(value=False, description='Large text', indent=False,
                         layout={'width': '110px'})

_xr_out = Output()
_xr_busy = {'v': False}   # suppress refreshes while the slider options are being rebuilt

def _xr_refresh():
    if _xr_busy['v']:
        return
    sk = _xr_stage_sel()
    mt = _xr_metric_d.value
    if not sk:
        return
    _rng = _xr_range_sl.value if _xr_range_chk.value else None
    _bars = XR_POOL_COLORS if _xr_pool_chk.value else None   # pooled → bar plot
    with _xr_out:
        _xr_out.clear_output(wait=True)
        if mt == 'all':
            for _s in plot_h_reflex_comparison_stacked(*_xr_src(), sk, _xr_stages,
                                                       avg_range=_rng,
                                                       show_box_text=_xr_boxtxt_chk.value,
                                                       bar_colors=_bars,
                                                       large_text=_xr_large_chk.value,
                                                       panel_height=5.0):   # ← add this
                if _s:
                    print(format_h_comparison_summary(_s) + '\n')
            return
        _xr_summary = plot_h_reflex_comparison(*_xr_src(), sk, _xr_stages, metric=mt,
                                 avg_range=_rng,
                                 compact=_xr_compact_chk.value,
                                 show_box_text=_xr_boxtxt_chk.value,
                                 bar_colors=_bars,
                                 large_text=_xr_large_chk.value)
        if _xr_summary:
            print(format_h_comparison_summary(_xr_summary))

def _xr_on_stage(change):
    # Range slider options follow the plotted order (recordings of the shown stage(s) only;
    # pools replace their member recordings when "Pool days" is on).
    _xr_busy['v'] = True
    try:
        _opts = _xr_entry_labels()
        _xr_range_sl.options = _opts
        _xr_range_sl.index = (0, len(_opts) - 1)
    finally:
        _xr_busy['v'] = False
    _xr_refresh()

def _xr_on_range_chk(change):
    _xr_range_sl.disabled = not change['new']
    _xr_refresh()

_xr_stage_d.observe(_xr_on_stage, names='value')
_xr_stage2_d.observe(_xr_on_stage, names='value')
_xr_pool_chk.observe(_xr_on_stage, names='value')
_xr_metric_d.observe(lambda c: _xr_refresh(), names='value')
_xr_range_chk.observe(_xr_on_range_chk, names='value')
_xr_range_sl.observe(lambda c: _xr_refresh(), names='value')
_xr_compact_chk.observe(lambda c: _xr_refresh(), names='value')
_xr_boxtxt_chk.observe(lambda c: _xr_refresh(), names='value')
_xr_large_chk.observe(lambda c: _xr_refresh(), names='value')
_disp(VBox([_xr_stage_d, _xr_stage2_d, _xr_metric_d,
            HBox([_xr_compact_chk, _xr_boxtxt_chk, _xr_large_chk, _xr_pool_chk]),
            HBox([_xr_range_chk, _xr_range_lbl, _xr_range_sl]), _xr_out]))
_xr_refresh()

# Section 6d: Cross-Recording DataFrame & Calculations

`trials_df` has one row per trial of every loaded recording and stage, **including excluded trials** (`included` = False). Its columns are:
- `trial` — the session trial number.
- The computed `h_size_uv`, `m_size_uv`, `bg_mra_uv`, `hm_ratio`. These are calculated the same way as the comparison plot (Section 6b): rectified window mean minus the standard peri-stim background, NaN when the background can't be measured.
- **Every scalar field the trial stores** — stim amplitude, polarity, onset info, the app's own H/M values (`h_wave_response`, `app_hm_ratio`), M-wave stabilizer fields, `data_integrity_flags`, and for conditioning stages `vns_delivered`, `success_threshold`, `applied_success_threshold`, `reward_threshold_used`, `decision_*`, `onset_to_vns_*_ms`, etc.

`xr_df` is the included trials only, which is what every plot above uses. Re-run these cells after re-running the pre-compute cell above.

In [47]:
import pandas as pd

# ── Per-trial DataFrame (every trial, every stage) ────────────────────────────
trials_df = build_trial_dataframe(
    _all_recordings_raw, _all_recordings,
    PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS, M_WAVE_START_MS, M_WAVE_END_MS,
)
xr_df = trials_df[trials_df['included']].reset_index(drop=True)
print(f"trials_df: {len(trials_df)} rows ({int((~trials_df['included']).sum())} excluded)  |  "
      f"{trials_df.shape[1]} columns")
print(f"xr_df:     {len(xr_df)} included trials  |  "
      f"recordings: {xr_df['recording'].nunique()}  |  stages: {xr_df['stage'].nunique()}")
if saved_analysis is not None:   # loaded from a pickle: are the analysis settings the same?
    diff_analysis_config(saved_analysis['config'], collect_notebook_config(globals()))
xr_df.head(10)

trials_df: 21014 rows (480 excluded)  |  65 columns
xr_df:     20534 included trials  |  recordings: 19  |  stages: 3
  config differs: ALSO_LOAD_RECORDING_DIRS  saved=False  now=True
  config differs: H_WAVE_END_MS  saved=9.5  now=10
  config differs: H_WAVE_START_MS  saved=5.5  now=6
  config differs: LOAD_ANALYSIS_PICKLE  saved=None  now='Up-Conditioning_VNS\\HRPilot-27\\HRPilot-27_h_reflex_analysis.pkl'
  config differs: RECORDING_DIRS  saved=[('-5', 'Control_Mode/HRPilot-27/NEW_SET-UP_TEST_HRPILOT-27_BOOTH4_100US_10KHZ_10-16-26', 10000.0), ('-4', 'Control_Mode/HRPilot-27/CMN2_HRPILOT-27_BOOTH4_100US_10KHZ_10-20-26', 10000.0), ('-3', 'Control_Mode/HRPilot-27/CMN3_HRPILOT-27_BOOTH4_100US_10KHZ_10-21-26', 10000.0), ('-2', 'Control_Mode/HRPilot-27/CMN4_HRPILOT-27_BOOTH4_100US_10KHZ_10-21-26', 10000.0), ('-1', 'Control_Mode/HRPilot-27/CMN5_HRPILOT-27_BOOTH4_100US_10KHZ_10-22-26', 10000.0), ('1', 'Up-Conditioning_VNS/HRPilot-27/UPVNS1_HRPILOT-27_BOOTH4_100US_10KHZ_10-22-26', 10000.0), (

,recording,stage,stage_key,trial,included,h_size_uv,m_size_uv,bg_mra_uv,hm_ratio,start_time,...,vns_delivered,success_threshold,is_success,vns_valid_sample_count,vns_detect_threshold_v,vns_adc_pulse_sample_index,vns_waveform_pulse_sample_index,onset_to_vns_adc_ms,onset_to_vns_waveform_ms,applied_success_threshold
0,-5,Control Mode (.hrs2),control_mode,2,True,35.089203,700.616028,155.110534,0.050083,2026-10-16 15:51:19.091185,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-5,Control Mode (.hrs2),control_mode,3,True,-21.148891,634.944401,149.323360,-0.033308,2026-10-16 15:51:24.320989,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-5,Control Mode (.hrs2),control_mode,4,True,-95.127928,761.443117,264.470823,-0.124931,2026-10-16 15:51:30.642037,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-5,Control Mode (.hrs2),control_mode,5,True,4.468664,1769.666601,322.931299,0.002525,2026-10-16 15:51:36.512404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-5,Control Mode (.hrs2),control_mode,6,True,49.016974,1247.238333,130.997385,0.039300,2026-10-16 15:51:55.552393,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,-5,Control Mode (.hrs2),control_mode,7,True,133.748724,1580.401434,237.818414,0.084630,2026-10-16 15:52:00.651400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,-5,Control Mode (.hrs2),control_mode,9,True,-57.645557,686.465527,203.569629,-0.083974,2026-10-16 15:52:53.602470,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,-5,Control Mode (.hrs2),control_mode,10,True,136.358238,1363.191963,116.788872,0.100029,2026-10-16 15:53:03.413099,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,-5,Control Mode (.hrs2),control_mode,11,True,39.872420,592.433501,159.254792,0.067303,2026-10-16 15:53:10.262770,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,-5,Control Mode (.hrs2),control_mode,12,True,108.941007,1306.492429,122.238772,0.083384,2026-10-16 15:53:28.313085,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# ── Per-recording summary statistics ─────────────────────────────────────────
xr_summary = (
    xr_df.groupby('recording')
    .agg(
        n_trials      = ('trial',      'count'),
        h_size_mean   = ('h_size_uv',  'mean'),
        h_size_std    = ('h_size_uv',  'std'),
        m_size_mean   = ('m_size_uv',  'mean'),
        m_size_std    = ('m_size_uv',  'std'),
        bg_mra_mean   = ('bg_mra_uv',  'mean'),
        hm_ratio_mean = ('hm_ratio',   'mean'),
        hm_ratio_std  = ('hm_ratio',   'std'),
    )
    .round(3)
)
xr_summary

,n_trials,h_size_mean,h_size_std,m_size_mean,m_size_std,bg_mra_mean,hm_ratio_mean,hm_ratio_std
recording,,,,,,,,
-1,817,170.310,180.226,971.277,386.484,183.485,0.242,0.348
-2,801,226.788,191.390,986.648,450.295,182.956,0.353,0.544
-3,989,168.638,178.673,1062.952,593.901,178.251,0.332,0.818
-4,1174,208.013,198.429,855.718,514.002,170.686,0.502,1.181
-5,1111,117.855,145.552,1251.281,782.480,155.835,0.158,0.733
1,937,182.092,186.509,1061.063,453.656,184.057,0.231,0.343
10,1174,185.223,202.961,1075.543,517.339,177.692,0.263,0.652
11,1025,191.777,195.027,1105.927,564.664,183.379,0.278,0.472
12,1099,278.552,228.643,1113.189,593.276,191.772,0.362,0.537


### Recording Settings DataFrame

`settings_df` captures the app-side stimulation/window settings that were active for each
(recording, stage) — parsed from the `*.settings.json` sidecar written next to each binary
data file by newer H-Reflex App recordings (sweep range, M/H-wave windows, thresholds,
polarity, pulse-train parameters, etc). Stages recorded before the app wrote sidecars still
get a row here; their settings columns are simply NaN.

In [49]:
# ── Build settings DataFrame from *.settings.json sidecars ───────────────────────
settings_df = build_settings_dataframe(_all_recordings)
_meta_cols = {'recording', 'stage', 'stage_key', 'subject_id', 'session_start',
              'app_version', 'file_version', 'sample_rate_hz', 'n_trials'}
_setting_cols = [c for c in settings_df.columns if c not in _meta_cols]
_n_with_settings = int(settings_df[_setting_cols].notna().any(axis=1).sum()) if _setting_cols else 0
print(f"settings_df: {len(settings_df)} rows  |  {settings_df['recording'].nunique()} recording(s)  |  "
      f"{_n_with_settings}/{len(settings_df)} stage(s) have a settings.json sidecar")
settings_df.head(10)

settings_df: 21 rows  |  19 recording(s)  |  20/21 stage(s) have a settings.json sidecar


,recording,stage,stage_key,subject_id,session_start,app_version,file_version,sample_rate_hz,n_trials,user_min_stimulation_amplitude,...,fixed_stimulation_amplitude_ma,filtering_protocol,n_pulses,event_period_us,event_period_us_b,inter_train_interval_ms,stimulation_amplitude_ma,pulse_width_us,user_stimulation_amplitude_step_size,sequential_stimulation
0,-5,Control Mode (.hrs2),control_mode,HRPILOT-27,2026-10-16 15:50:51.068846,H-Reflex App Version 3,18,55000.0,1004,0.15500,...,0.15500,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-5,Frequency Test (.hrft),frequency_test,HRPILOT-27,2026-10-17 19:55:04.296649,H-Reflex App Version 3,12,10000.0,107,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-4,Control Mode (.hrs2),control_mode,HRPILOT-27,2026-10-20 08:13:40.018408,H-Reflex App Version 3,18,55000.0,1174,0.20500,...,0.20500,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-3,Control Mode (.hrs2),control_mode,HRPILOT-27,2026-10-21 09:10:16.559400,H-Reflex App Version 3,18,55000.0,880,0.20500,...,0.20500,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-3,Frequency Test (.hrft),frequency_test,HRPILOT-27,2026-10-21 14:02:53.967567,H-Reflex App Version 3,12,10000.0,109,NaN,...,NaN,OFFLINE,5.0,3333333.0,66667.0,10000.0,0.15,100.0,NaN,NaN
5,-2,Control Mode (.hrs2),control_mode,HRPILOT-27,2026-10-21 14:44:26.549581,H-Reflex App Version 3,18,55000.0,801,0.17000,...,0.17000,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,-1,Control Mode (.hrs2),control_mode,HRPILOT-27,2026-10-22 08:11:40.207038,H-Reflex App Version 3,18,55000.0,817,0.17999,...,0.17999,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,Up Condition VNS (.hrs6),up_cond_vns,HRPILOT-27,2026-10-22 12:44:46.223863,H-Reflex App Version 3,14,10000.0,937,0.18000,...,NaN,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,0.1,False
8,2,Up Condition VNS (.hrs6),up_cond_vns,HRPILOT-27,2026-10-23 09:28:50.210370,H-Reflex App Version 3,14,10000.0,1096,0.18000,...,NaN,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,0.1,False
9,3,Up Condition VNS (.hrs6),up_cond_vns,HRPILOT-27,2026-10-26 11:08:10.538662,H-Reflex App Version 3,15,10000.0,954,0.13000,...,NaN,OFFLINE,NaN,NaN,NaN,NaN,NaN,NaN,0.1,False


### Save Analysis to Disk

Set `SAVE_ANALYSIS = True` and run the cell to write a snapshot. Nothing is saved automatically. The pickle contains:
- `trials_df`, `xr_df`, `xr_summary`, `settings_df`, `excluded_trials_df`, `prefilter_metrics_df`, and the comparison-plot caches (`xr_cache`, `xr_pool_cache`)
- `config`: every UPPER_CASE setting in the notebook (M/H windows, PRE/POST ms, `RECORDING_DIRS`, exclusion and pre-filter settings, `XR_POOLS`, …)
- with `INCLUDE_RECORDINGS = True`: the loaded recordings (raw and cleaned, all trial data and headers), so setting `LOAD_ANALYSIS_PICKLE` in the load cell restores the whole notebook without the binary files

The raw continuous EMG blocks are left out unless `INCLUDE_EMG_BLOCKS = True`, because they are most of each recording's size. Without them, the BG-EMG monitoring window, the filtering viewer and the failed-trial alignment check have no data. `trial_data` already holds the OFFLINE-reconstructed EMG, so every other plot is unaffected.

To open a snapshot without the notebook: `from helpers import load_analysis_pickle; s = load_analysis_pickle("file.pkl"); s["trials_df"]`.

In [50]:
# ── Save an analysis snapshot to a pickle file ────────────────────────────────
SAVE_ANALYSIS      = True   # True → write the file when this cell runs
ANALYSIS_PKL_PATH  = "HRPilot-27_h_reflex_analysis.pkl"  # None → h_reflex_analysis_<timestamp>.pkl in this folder
INCLUDE_RECORDINGS = True    # trial data + headers, so the notebook can be rebuilt from the pickle
INCLUDE_EMG_BLOCKS = False   # raw continuous EMG too (many GB per recording)

if SAVE_ANALYSIS:
    save_analysis_pickle(
        ANALYSIS_PKL_PATH,
        config=collect_notebook_config(globals()),
        all_recordings_raw=_all_recordings_raw, all_recordings=_all_recordings,
        include_recordings=INCLUDE_RECORDINGS, include_emg_blocks=INCLUDE_EMG_BLOCKS,
        trials_df=trials_df, xr_df=xr_df, xr_summary=xr_summary, settings_df=settings_df,
        excluded_trials_df=excluded_trials_df, prefilter_metrics_df=prefilter_metrics_df,
        xr_cache=_xr_cache, xr_pool_cache=_xr_pool_cache,
    )
else:
    print("SAVE_ANALYSIS = False — set it to True and re-run this cell to write the snapshot.")

Saved 2321.3 MB → HRPilot-27_h_reflex_analysis.pkl
  contents: ['format', 'meta', 'config', 'trials_df', 'xr_df', 'xr_summary', 'settings_df', 'excluded_trials_df', 'xr_cache', 'xr_pool_cache', 'recordings_raw', 'recordings']


In [29]:
# ── Custom calculations — edit this cell freely ───────────────────────────────
# Examples:

# Filter to a specific recording
# rec = xr_df[xr_df['recording'] == 'HRPILOT-23']

# Mean H-size per recording (µV)
# xr_df.groupby('recording')['h_size_uv'].mean()

# Coefficient of variation (%) for H-size per recording
# xr_df.groupby('recording')['h_size_uv'].agg(lambda x: x.std() / x.mean() * 100).rename('h_cv_pct')

# Export to CSV
# xr_df.to_csv('xr_data.csv', index=False)
# xr_summary.to_csv('xr_summary.csv')

print("xr_df and xr_summary are ready.  Edit this cell to run your calculations.")

xr_df and xr_summary are ready.  Edit this cell to run your calculations.
